# 3AM 推理 Notebook

这个 notebook 走一条手动交互的推理路径：

1. 载入训练配置和 checkpoint。
2. 读取视频或帧目录，选择参考帧。
3. 在参考帧上输入前景/背景点，先用 SAM2 生成 seed mask。
4. 把 seed mask 传播到整段视频，保存逐帧 mask 和 mp4。

`POINT_LABELS = 1` 表示前景点，`0` 表示背景点。

In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').exists() and (candidate / 'scripts').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root starting from {start}')

ROOT = find_repo_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from three_am.models.adapters import Must3rFeatureAdapter
from three_am.training.dataset import load_image_tensor
from three_am.utils.config import load_yaml

def load_script_module(script_path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Could not load {script_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

train_3am = load_script_module(ROOT / 'scripts' / 'train_3am.py', 'train_3am_notebook')
infer_3am = load_script_module(ROOT / 'scripts' / 'infer_3am_tracking_video.py', 'infer_3am_tracking_video_notebook')
print(ROOT)

In [ ]:
CONFIG = ROOT / 'configs' / 'full_reproduction.yaml'
CHECKPOINT = ROOT / 'outputs' / 'checkpoints' / '3am_full.pt'
VIDEO = ROOT / 'path' / 'to' / 'video.mp4'  # 也可以是帧目录
OUTPUT_DIR = ROOT / 'outputs' / 'infer' / 'demo'

REFERENCE_INDEX = 0
POINTS = [(100.0, 100.0)]  # 原图像素坐标，手动改这里
POINT_LABELS = [1]  # 1=前景点，0=背景点

IMAGE_SIZE = 1024
THRESHOLD = 0.5
FPS = 24
CHUNK_SIZE = 32  # 0=关闭 chunk；更长视频建议保持 > 0
SAVE_LETTERBOX_MASKS = False

assert len(POINTS) == len(POINT_LABELS) and len(POINTS) > 0

In [ ]:
config = load_yaml(CONFIG)
device = train_3am._device(None)
wrapper = infer_3am._load_wrapper(config, CHECKPOINT, device)
must3r_adapter = Must3rFeatureAdapter(train_3am.external_config(config, must3r_device=str(device)))

frame_paths = infer_3am._frame_paths(VIDEO)
if not frame_paths:
    raise ValueError('No frames found')
if not 0 <= REFERENCE_INDEX < len(frame_paths):
    raise ValueError(f'REFERENCE_INDEX must be in [0, {len(frame_paths) - 1}]')

source_shape = infer_3am._source_shape(frame_paths[0])
transform = infer_3am._letterbox_transform(
    source_width=source_shape[1],
    source_height=source_shape[0],
    target_size=IMAGE_SIZE,
)
reference_path = frame_paths[REFERENCE_INDEX]
reference_raw = load_image_tensor(reference_path, None)
reference_canvas = infer_3am._load_images_with_transform([reference_path], transform).to(device)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'frames={len(frame_paths)} source_shape={source_shape} device={device}')

In [ ]:
def points_to_canvas(points, transform):
    return torch.tensor(
        [transform.resize_point(float(x), float(y)) for x, y in points],
        dtype=torch.float32,
        device=device,
    )

def image_tensor_to_numpy(image_tensor):
    return image_tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()

def show_reference_points(image_tensor, points, labels, title):
    array = image_tensor_to_numpy(image_tensor)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(array)
    for (x, y), label in zip(points, labels):
        color = 'lime' if int(label) == 1 else 'red'
        ax.scatter([x], [y], s=70, c=color, edgecolors='white', linewidths=1.2)
        ax.text(
            x + 4,
            y + 4,
            str(int(label)),
            color='white',
            fontsize=10,
            bbox=dict(facecolor='black', alpha=0.35, pad=1, edgecolor='none'),
        )
    ax.set_title(title)
    ax.axis('off')
    plt.show()

def overlay_mask(image_path, mask_path, color=(255, 64, 0), alpha=0.55):
    image = Image.open(image_path).convert('RGB')
    mask = Image.open(mask_path).convert('L')
    if mask.size != image.size:
        mask = mask.resize(image.size, resample=Image.Resampling.NEAREST)
    image_np = np.asarray(image, dtype=np.uint8)
    mask_np = np.asarray(mask) > 0
    overlay = image_np.astype(np.float32)
    color_np = np.asarray(color, dtype=np.float32)
    overlay[mask_np] = overlay[mask_np] * (1.0 - alpha) + color_np * alpha
    return overlay.clip(0, 255).astype(np.uint8)

show_reference_points(reference_raw, POINTS, POINT_LABELS, f'reference frame {REFERENCE_INDEX}')

In [ ]:
point_coords = points_to_canvas(POINTS, transform)
point_labels = torch.tensor(POINT_LABELS, dtype=torch.int32, device=device)
seed_logits = wrapper.sam2_adapter.predict_masks_from_points(
    reference_canvas,
    [point_coords],
    [point_labels],
)
seed_prob = seed_logits[0].sigmoid().detach().cpu()
seed_mask = (seed_prob >= THRESHOLD).float()
seed_pixels = int(seed_mask.sum().item())
if seed_pixels == 0:
    raise ValueError('SAM2 point prompt produced an empty seed mask. Adjust POINTS/POINT_LABELS or lower THRESHOLD.')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image_tensor_to_numpy(reference_raw))
for (x, y), label in zip(POINTS, POINT_LABELS):
    color = 'lime' if int(label) == 1 else 'red'
    axes[0].scatter([x], [y], s=70, c=color, edgecolors='white', linewidths=1.2)
axes[0].set_title('reference + points')
axes[0].axis('off')
axes[1].imshow(seed_prob.numpy(), cmap='magma')
axes[1].set_title('seed probability')
axes[1].axis('off')
axes[2].imshow(seed_mask.numpy(), cmap='gray')
axes[2].set_title('seed mask')
axes[2].axis('off')
plt.tight_layout()

In [ ]:
mask_dir = OUTPUT_DIR / 'masks'
mask_video_path = OUTPUT_DIR / 'masks.mp4'
output_transform = None if SAVE_LETTERBOX_MASKS else transform
output_source_shape = None if SAVE_LETTERBOX_MASKS else source_shape
if CHUNK_SIZE < 0:
    raise ValueError('CHUNK_SIZE must be 0 or a positive integer')
if 0 < CHUNK_SIZE < 2:
    raise ValueError('CHUNK_SIZE must be 0 or at least 2')
chunked = CHUNK_SIZE > 0 and len(frame_paths) > CHUNK_SIZE

if chunked:
    saved_by_index = infer_3am._run_chunked_inference(
        frame_paths=frame_paths,
        video_path=VIDEO,
        reference_index=REFERENCE_INDEX,
        reference_mask=seed_mask,
        transform=transform,
        wrapper=wrapper,
        must3r_adapter=must3r_adapter,
        device=device,
        chunk_size=CHUNK_SIZE,
        mask_dir=mask_dir,
        threshold=THRESHOLD,
        output_transform=output_transform,
        output_source_shape=output_source_shape,
    )
    mask_paths = [saved_by_index[index] for index in range(len(frame_paths))]
else:
    images = infer_3am._load_images_with_transform(frame_paths, transform)
    batch = infer_3am._build_inference_batch(
        images=images.to(device),
        frame_paths=frame_paths,
        prompt_mask=seed_mask.to(device),
        reference_index=REFERENCE_INDEX,
        video_path=VIDEO,
        transform=transform,
        sampling_mode='full_scene',
    )
    logits = infer_3am._predict_logits(
        wrapper=wrapper,
        must3r_adapter=must3r_adapter,
        batch=batch,
        device=device,
    )
    mask_paths = infer_3am._save_masks(
        logits,
        mask_dir,
        threshold=THRESHOLD,
        transform=output_transform,
        source_shape=output_source_shape,
    )

infer_3am._write_video(mask_dir, mask_video_path, fps=FPS)
assert len(mask_paths) == len(frame_paths)

summary = {
    'frames': len(frame_paths),
    'reference_index': REFERENCE_INDEX,
    'reference_frame': str(reference_path),
    'output_dir': str(OUTPUT_DIR),
    'mask_dir': str(mask_dir),
    'mask_video': str(mask_video_path),
    'mask_count': len(mask_paths),
    'chunk_size': CHUNK_SIZE,
    'chunked': chunked,
    'seed_mask_shape': tuple(seed_mask.shape),
    'source_shape': source_shape,
}
summary

In [ ]:
preview_indices = sorted({REFERENCE_INDEX, min(REFERENCE_INDEX + 1, len(frame_paths) - 1), len(frame_paths) - 1})
fig, axes = plt.subplots(1, len(preview_indices), figsize=(6 * len(preview_indices), 5))
axes = np.atleast_1d(axes)
for ax, index in zip(axes, preview_indices):
    ax.imshow(overlay_mask(frame_paths[index], mask_paths[index]))
    ax.set_title(f'frame {index}')
    ax.axis('off')
plt.tight_layout()